# 00 — Setup & Dataset Download

Downloads VisDrone2019-DET from Kaggle, converts annotations to YOLO format,
and organizes files into the project structure expected by all experiments.

**Steps:**
1. Verify Kaggle authentication
2. Download dataset (`kushagrapandya/visdrone-dataset`)
3. Extract and reorganize into `data/VisDrone2019-DET/`
4. Convert VisDrone → YOLO annotation format
5. Validate final structure
6. Save `results/setup_report.json`

**Prerequisites:** Kaggle API token configured (see Cell 2).


In [1]:
import os
import sys
import shutil
import subprocess
import zipfile
from pathlib import Path

# Ensure working directory is project root
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
os.chdir(root)
print(f"Working directory: {Path.cwd()}")

# Add to sys.path for src imports
if str(root) not in sys.path:
    sys.path.insert(0, str(root))


Working directory: /Users/davide/Desktop/Universita/AI/CV_DL/project


## 1. Kaggle Authentication

VisDrone dataset downloaded via Kaggle API. Requires Kaggle token.

**Setup (one-time):**
```bash
# Option A: New token file
mkdir -p ~/.kaggle && echo YOUR_TOKEN > ~/.kaggle/access_token && chmod 600 ~/.kaggle/access_token

# Option B: Legacy kaggle.json (download from kaggle.com → Settings → API)
mv ~/Downloads/kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
```


In [2]:
home = Path.home()
token_new = home / ".kaggle" / "access_token"
token_old = home / ".kaggle" / "kaggle.json"

if token_new.exists():
    print(f"[OK] Token found: {token_new}")
    perms = oct(token_new.stat().st_mode)[-3:]
    if perms != "600":
        print(f"[WARN] Permissions are {perms}, expected 600. Fix:")
        print(f"       chmod 600 {token_new}")
elif token_old.exists():
    print(f"[OK] Token found: {token_old}")
    perms = oct(token_old.stat().st_mode)[-3:]
    if perms != "600":
        print(f"[WARN] Permissions are {perms}, expected 600. Fix:")
        print(f"       chmod 600 {token_old}")
else:
    raise FileNotFoundError(
        "Kaggle token not found!\n"
        "Create ~/.kaggle/access_token or ~/.kaggle/kaggle.json\n"
        "See instructions above."
    )

# Quick auth test: list datasets (does not download anything)
result = subprocess.run(
    ["kaggle", "datasets", "list", "--max-size", "1"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("[OK] Kaggle authentication works.")
else:
    print("[FAIL] Kaggle authentication failed:")
    print(result.stderr)
    raise ConnectionError("Kaggle auth failed. Check your token.")


[OK] Token found: /Users/davide/.kaggle/access_token
[OK] Kaggle authentication works.


## 2. Download VisDrone2019-DET from Kaggle

Downloads `kushagrapandya/visdrone-dataset` — contains:
- `VisDrone2019-DET-train/` — 6,471 images
- `VisDrone2019-DET-val/` — 548 images
- `VisDrone2019-DET-test-dev/` — 1,610 images
- `VisDrone.yaml` — Ultralytics config

Annotations in original VisDrone format (comma-separated). Converted to YOLO in Step 3.


In [3]:
data_dir = root / "data"
kaggle_zip = data_dir / "visdrone-dataset.zip"
extract_dir = data_dir / "_kaggle_extract"
target_dir = data_dir / "VisDrone2019-DET"

KAGGLE_DATASET = "kushagrapandya/visdrone-dataset"
SPLITS = ["VisDrone2019-DET-train", "VisDrone2019-DET-val", "VisDrone2019-DET-test-dev"]

# Check if dataset is already complete
train_img_dir = target_dir / "images" / "train"
if train_img_dir.exists():
    n_train = len(list(train_img_dir.glob("*")))
else:
    n_train = 0

dataset_ready = n_train >= 6471

if dataset_ready:
    print(f"[SKIP] Dataset already ready at {target_dir} ({n_train} train images)")
else:
    if kaggle_zip.exists():
        print(f"[SKIP] ZIP already exists: {kaggle_zip} ({kaggle_zip.stat().st_size / 1e6:.1f} MB)")
    else:
        print(f"[DOWNLOAD] {KAGGLE_DATASET} → {kaggle_zip}")
        result = subprocess.run(
            ["kaggle", "datasets", "download", KAGGLE_DATASET, "-p", str(data_dir)],
            capture_output=True, text=True
        )
        if result.returncode != 0:
            print(result.stderr)
            raise RuntimeError("Kaggle download failed.")
        zip_size = kaggle_zip.stat().st_size / 1e6
        print(f"[OK] Download complete ({zip_size:.1f} MB)")


[SKIP] Dataset already ready at /Users/davide/Desktop/Universita/AI/CV_DL/project/data/VisDrone2019-DET (6471 train images)


### Extract & Reorganize

Unzips Kaggle archive and reorganizes into project structure:
```
data/VisDrone2019-DET/
├── images/{train,val,test-dev}/   ← images
├── annotations/{train,val,test-dev}/  ← original VisDrone annotations
└── labels/{train,val,test-dev}/   ← YOLO-format labels (converted next)
```


In [4]:
# Safety: init in case cell ran standalone (without Cell 6)
if "dataset_ready" not in dir():
    train_img_dir = target_dir / "images" / "train"
    n_train = len(list(train_img_dir.glob("*"))) if train_img_dir.exists() else 0
    dataset_ready = n_train >= 6471

if dataset_ready:
    print("[SKIP] Dataset already organized, nothing to extract.")

elif kaggle_zip.exists():
    print(f"[EXTRACT] {kaggle_zip} → {extract_dir}")
    with zipfile.ZipFile(kaggle_zip, "r") as zf:
        zf.extractall(extract_dir)
    print("[OK] Extraction complete.")

    # ZIP internal structure is double-nested:
    #   VisDrone2019-DET-train/VisDrone2019-DET-train/images/
    #   VisDrone2019-DET-train/VisDrone2019-DET-train/annotations/
    for split in SPLITS:
        split_name = split.replace("VisDrone2019-DET-", "")  # train, val, test-dev
        # Double nesting: split/split/images
        src = extract_dir / split / split

        # Create target dirs
        for subdir in ["images", "annotations"]:
            dst = target_dir / subdir / split_name
            dst.mkdir(parents=True, exist_ok=True)

        # Copy images
        img_src = src / "images"
        img_dst = target_dir / "images" / split_name
        if img_src.exists():
            files = list(img_src.iterdir())
            for f in files:
                shutil.copy2(f, img_dst / f.name)
            print(f"  images/{split_name}: {len(files)} files")
        else:
            print(f"  [WARN] images/{split_name}: source not found at {img_src}")

        # Copy annotations (original VisDrone comma-separated format)
        ann_src = src / "annotations"
        ann_dst = target_dir / "annotations" / split_name
        if ann_src.exists():
            files = list(ann_src.iterdir())
            for f in files:
                shutil.copy2(f, ann_dst / f.name)
            print(f"  annotations/{split_name}: {len(files)} files")
        else:
            print(f"  [WARN] annotations/{split_name}: source not found at {ann_src}")

    # Clean up extraction dir
    shutil.rmtree(extract_dir, ignore_errors=True)
    print("[OK] Reorganization complete.")

else:
    print("[SKIP] No ZIP found — run download cell first (Cell 6).")


[SKIP] Dataset already organized, nothing to extract.


## 3. Convert Annotations: VisDrone → YOLO format

VisDrone annotations are comma-separated: `x1,y1,w,h,score,class_id,occlusion,truncation`
Convert to YOLO format: `class_id cx cy w h` (normalized to [0,1]).

Skip class 0 (ignored regions). Shift class IDs by -1 (VisDrone classes are 1-indexed).


In [5]:
from PIL import Image
from tqdm.notebook import tqdm

# Safety: re-init in case cell ran standalone
if "SPLITS" not in dir():
    SPLITS = ["VisDrone2019-DET-train", "VisDrone2019-DET-val", "VisDrone2019-DET-test-dev"]
if "target_dir" not in dir():
    target_dir = Path.cwd().parent / "data" / "VisDrone2019-DET" if Path.cwd().name != "project" else Path.cwd() / "data" / "VisDrone2019-DET"


def visdrone2yolo(split_dir: Path):
    """
    Convert VisDrone annotations to YOLO+ format for one split.

    VisDrone format per row (comma-separated):
        x1,y1,w,h,score,class_id,truncation,occlusion

    Where:
        class_id: 1-indexed (1=pedestrian..10=motor)
        score: 0 = ignored region (skip these)
        truncation: 0=no, 1=yes
        occlusion: 0=visible, 1=partial, 2=heavy

    Output YOLO+ format (7 columns — standard YOLO + occlusion/truncation):
        class_id cx cy w h occlusion truncation
    """
    ann_dir = split_dir / "annotations"
    labels_dir = split_dir / "labels"
    labels_dir.mkdir(parents=True, exist_ok=True)
    img_dir = split_dir / "images"

    ann_files = sorted(ann_dir.glob("*.txt"))
    if not ann_files:
        print(f"  [WARN] No annotations found in {ann_dir}")
        return 0

    total = 0
    for ann_file in tqdm(ann_files, desc=f"Converting {split_dir.name}"):
        # Get image size from corresponding image
        img_file = img_dir / ann_file.with_suffix(".jpg").name
        if not img_file.exists():
            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = img_dir / ann_file.with_suffix(ext).name
                if candidate.exists():
                    img_file = candidate
                    break
        if not img_file.exists():
            continue

        img = Image.open(img_file)
        w_img, h_img = img.size

        lines = []
        with open(ann_file) as f:
            for row in f.read().strip().splitlines():
                parts = row.split(",")
                if len(parts) < 8:
                    continue
                x1, y1, bw, bh, score, cls_id = map(int, parts[:6])
                # VisDrone format: parts[6]=truncation, parts[7]=occlusion
                truncation = int(parts[6])
                occlusion  = int(parts[7])
                # Skip ignored regions (score=0)
                if score == 0:  # VisDrone convention: score=0 means ignore
                    continue
                # Convert bbox to normalized YOLO format
                cx = (x1 + bw / 2) / w_img
                cy = (y1 + bh / 2) / h_img
                nw = bw / w_img
                nh = bh / h_img
                yolo_cls = cls_id - 1  # VisDrone is 1-indexed
                # 7-column format: YOLO + occlusion truncation
                lines.append(f"{yolo_cls} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f} {occlusion} {truncation}\n")
                total += 1

        # Write label file
        label_file = labels_dir / ann_file.name
        with open(label_file, "w") as f:
            f.writelines(lines)

    return total


# Backup originals → annotations_raw/ before overwriting
backup_dir = target_dir / "annotations_raw"
if not backup_dir.exists():
    print("Backing up original VisDrone annotations → annotations_raw/ ...")
    for split in SPLITS:
        split_name = split.replace("VisDrone2019-DET-", "")
        src = target_dir / "annotations" / split_name
        dst = backup_dir / split_name
        if src.exists():
            shutil.copytree(src, dst)
    print("[OK] Backup complete.")
else:
    print("[SKIP] annotations_raw/ already exists.")

# Run conversion for all splits
print("\nConverting annotations to YOLO+ format (7 columns)...")
total_boxes = 0
for split in SPLITS:
    split_name = split.replace("VisDrone2019-DET-", "")
    split_images = target_dir / "images" / split_name
    split_annotations = target_dir / "annotations" / split_name

    # Create temp combined dir so visdrone2yolo finds images/ + annotations/ siblings
    temp_split = target_dir / f"_{split_name}_tmp"
    temp_split.mkdir(parents=True, exist_ok=True)
    (temp_split / "images").symlink_to(split_images)
    (temp_split / "annotations").symlink_to(split_annotations)

    n = visdrone2yolo(temp_split)
    total_boxes += n

    # Copy YOLO+ labels → overwrite annotations dir
    label_src = temp_split / "labels"
    ann_dst = target_dir / "annotations" / split_name
    if label_src.exists():
        for f in label_src.iterdir():
            shutil.copy2(f, ann_dst / f.name)

    # Clean up temp
    shutil.rmtree(temp_split, ignore_errors=True)
    print(f"  {split_name}: {n} boxes converted")

print(f"\n[OK] Total: {total_boxes} bounding boxes in YOLO+ format (7 columns)")
print("    Columns: class_id cx cy w h occlusion truncation")
print("    Originals preserved in annotations_raw/")


[SKIP] annotations_raw/ already exists.

Converting annotations to YOLO+ format (7 columns)...


Converting _train_tmp:   0%|          | 0/6471 [00:00<?, ?it/s]

  train: 0 boxes converted


Converting _val_tmp:   0%|          | 0/548 [00:00<?, ?it/s]

  val: 0 boxes converted


Converting _test-dev_tmp:   0%|          | 0/1610 [00:00<?, ?it/s]

  test-dev: 0 boxes converted

[OK] Total: 0 bounding boxes in YOLO+ format (7 columns)
    Columns: class_id cx cy w h occlusion truncation
    Originals preserved in annotations_raw/


## 4. Validate Final Structure

Verify images + YOLO annotations present for train, val, and test-dev.


In [6]:
import json
import random
from datetime import datetime

# Safety: re-init target_dir in case cell ran standalone
if "target_dir" not in dir():
    target_dir = Path.cwd().parent / "data" / "VisDrone2019-DET" if Path.cwd().name != "project" else Path.cwd() / "data" / "VisDrone2019-DET"
if "root" not in dir():
    root = target_dir.parent.parent

# Define expected splits and counts
EXPECTED = {
    "train":    {"images": 6471, "annotations": 6471},
    "val":      {"images": 548,  "annotations": 548},
    "test-dev": {"images": 1610, "annotations": 1610},
}

results = {}
all_ok = True

print("Final structure validation:\n")
for split_name, expected in EXPECTED.items():
    img_dir = target_dir / "images" / split_name
    ann_dir = target_dir / "annotations" / split_name

    n_imgs = len(list(img_dir.glob("*"))) if img_dir.exists() else 0
    n_anns = len(list(ann_dir.glob("*.txt"))) if ann_dir.exists() else 0

    img_ok = n_imgs >= expected["images"] * 0.9
    ann_ok = n_anns >= expected["annotations"] * 0.9

    results[split_name] = {
        "images_found": n_imgs,
        "images_expected": expected["images"],
        "annotations_found": n_anns,
        "annotations_expected": expected["annotations"],
        "images_ok": img_ok,
        "annotations_ok": ann_ok,
    }

    status = "[OK]" if (img_ok and ann_ok) else "[FAIL]"
    if not (img_ok and ann_ok):
        all_ok = False

    print(f"  {status} {split_name}:")
    print(f"       images: {n_imgs}/{expected['images']}")
    print(f"       annotations: {n_anns}/{expected['annotations']}")

# Validate YOLO+ format on sample
# Columns: cls cx cy w h occlusion(0/1/2) truncation(0/1)
print("\nValidating YOLO+ format (50 random annotations)...")
val_ann_dir = target_dir / "annotations" / "val"
val_anns = sorted(val_ann_dir.glob("*.txt"))
if val_anns:
    sample_files = random.sample(val_anns, min(50, len(val_anns)))
else:
    sample_files = []
format_errors = []
for f in sample_files:
    with open(f) as fh:
        for line in fh:
            parts = line.strip().split()
            if len(parts) not in (5, 7):
                format_errors.append(f"{f.name}: expected 5 or 7 tokens, got {len(parts)}")
                continue
            try:
                vals = list(map(float, parts))
                cls, cx, cy, w, h = vals[:5]
                assert 0 <= int(cls) <= 9, f"class {cls} out of range"
                assert 0 <= cx <= 1
                assert 0 <= cy <= 1
                assert 0 < w <= 1
                assert 0 < h <= 1
                # Validate extra columns if present
                if len(parts) == 7:
                    occlusion  = int(vals[5])  # col 6: occlusion (0,1,2)
                    truncation = int(vals[6])  # col 7: truncation (0,1)
                    assert occlusion in (0, 1, 2), f"occlusion {occlusion} not in (0,1,2)"
                    assert truncation in (0, 1), f"truncation {truncation} not in (0,1)"
            except (ValueError, AssertionError) as e:
                format_errors.append(f"{f.name}: {e}")

if format_errors:
    print(f"  [FAIL] {len(format_errors)} format errors:")
    for e in format_errors[:5]:
        print(f"     {e}")
    all_ok = False
else:
    n_checked = len(sample_files)
    print(f"  [OK] All {n_checked} samples valid YOLO+ format (5 or 7 columns)")

print(f"\n{'[OK] ALL CHECKS PASSED' if all_ok else '[FAIL] SOME CHECKS FAILED'}")

# Save setup report
setup_report = {
    "timestamp": datetime.now().isoformat(),
    "project_root": str(root),
    "dataset_path": str(target_dir),
    "splits": results,
    "all_ok": all_ok,
    "annotation_format": "yolo_plus",
    "annotation_columns": 7,
    "classes": [
        "pedestrian", "people", "bicycle", "car", "van",
        "truck", "tricycle", "awning-tricycle", "bus", "motor",
    ],
}

report_path = root / "results" / "setup_report.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
with open(report_path, "w") as f:
    json.dump(setup_report, f, indent=2)

print(f"\n[OK] Setup report saved: {report_path}")


Final structure validation:

  [OK] train:
       images: 6471/6471
       annotations: 6471/6471
  [OK] val:
       images: 548/548
       annotations: 548/548
  [OK] test-dev:
       images: 1610/1610
       annotations: 1610/1610

Validating YOLO+ format (50 random annotations)...
  [OK] All 50 samples valid YOLO+ format (5 or 7 columns)

[OK] ALL CHECKS PASSED

[OK] Setup report saved: /Users/davide/Desktop/Universita/AI/CV_DL/project/results/setup_report.json
